# Standard errors and tests

The quadratic fit $1 + 2x + 3x^2$ has residual sum $10$ on $2$ degrees of freedom. The inverse of $X^{\mathsf T}X$ turns that variance estimate into a standard error for each coefficient. With normal errors, the ratio of a coefficient to its standard error is a $t_2$ random variable when the tested value is the true one.


## The covariance matrix

$$
s^2 = \dfrac{\mathrm{RSS}}{n - p} = \dfrac{10}{2} = 5.
$$

The Gram matrix and its inverse are

$$
X^{\mathsf T}X = \begin{bmatrix} 5 & 0 & 10 \\ 0 & 10 & 0 \\ 10 & 0 & 34 \end{bmatrix}, \qquad
(X^{\mathsf T}X)^{-1} = \begin{bmatrix}
17/35 & 0 & -1/7 \\
0 & 1/10 & 0 \\
-1/7 & 0 & 1/14
\end{bmatrix}.
$$

The even block $\bigl[\begin{smallmatrix} 5 & 10 \\ 10 & 34 \end{smallmatrix}\bigr]$ has determinant $70$, and

$$
\dfrac{1}{70}\begin{bmatrix} 34 & -10 \\ -10 & 5 \end{bmatrix}
= \begin{bmatrix} 17/35 & -1/7 \\ -1/7 & 1/14 \end{bmatrix}
$$

is that corner of the inverse. Multiply by $s^2 = 5$:

$$
\begin{aligned}
\operatorname{se}(b_0) &= \sqrt{5\cdot\dfrac{17}{35}} = \sqrt{\dfrac{17}{7}}, \\
\operatorname{se}(b_1) &= \sqrt{5\cdot\dfrac{1}{10}} = \sqrt{\dfrac{1}{2}}, \\
\operatorname{se}(b_2) &= \sqrt{5\cdot\dfrac{1}{14}} = \sqrt{\dfrac{5}{14}}.
\end{aligned}
$$

The $t$ statistics for the hypotheses that each coefficient equals zero are

$$
\begin{aligned}
t_0 &= \dfrac{1}{\sqrt{17/7}} = \sqrt{\dfrac{7}{17}}, \\
t_1 &= \dfrac{2}{\sqrt{1/2}} = \sqrt{8}, \\
t_2 &= \dfrac{3}{\sqrt{5/14}} = \sqrt{\dfrac{126}{5}}.
\end{aligned}
$$

Numerically these are about $0.642$, $2.828$, and $5.020$. The $0.975$ quantile of $t_2$ is about $4.303$. Only $t_2$ exceeds it. The $95\%$ interval for the curvature is

$$
3 \pm t_{2,\,0.975}\sqrt{\dfrac{5}{14}}.
$$

Its square recovers the extra-sum-of-squares test: $t_2^2 = 126/5 = F$. The interval excludes zero. The intervals for $b_0$ and $b_1$ include zero. With two residual degrees of freedom, a coefficient must be large compared with $s$ before the test rejects.


In [1]:
# Standard errors, t statistics, and the interval for the quadratic coefficient.
import math
import scipy.stats as stats
se0 = math.sqrt(17 / 7)
se1 = math.sqrt(1 / 2)
se2 = math.sqrt(5 / 14)
t0 = 1 / se0
t1 = 2 / se1
t2 = 3 / se2
critical = stats.t.ppf(0.975, 2)
lower, upper = 3 - critical * se2, 3 + critical * se2
print("se", se0, se1, se2)
print("t", t0, t1, t2)
print("interval for b2", lower, upper)
assert abs(t2 - math.sqrt(126 / 5)) < 1e-12
assert abs(t1 - math.sqrt(8)) < 1e-12
assert abs(t0 - math.sqrt(7 / 17)) < 1e-12
assert abs(t2 ** 2 - 126 / 5) < 1e-12
assert t0 < critical and t1 < critical < t2
assert lower > 0


se 1.558387444947959 0.7071067811865476 0.5976143046671968
t 0.641688947919748 2.82842712474619 5.019960159204453
interval for b2 0.42867318068635907 5.571326819313641


## One change: test the value the polynomial actually used

The table was built so that the quadratic signal, before the cubic residual, had curvature $3$. The statistic for $H_0\!: b_2 = 3$ is

$$
t = \dfrac{3 - 3}{\sqrt{5/14}} = 0.
$$

The interval centered at the estimate $3$ contains $3$. A computing error that produced an interval excluding the fitted value would be visible immediately. The interesting exclusion is zero, not the center of the interval.


In [2]:
# The interval for b2 contains the fitted value 3 and excludes 0.
import math
import scipy.stats as stats
half = stats.t.ppf(0.975, 2) * math.sqrt(5 / 14)
print("half width", half)
assert 3 - half < 3 < 3 + half
assert 3 - half > 0


half width 2.571326819313641


## Pitfall

Failing to reject $b_1 = 0$ does not mean the slope is absent from $1 + 2x + 3x^2$. The estimate is $2$, and $\sqrt{8} \approx 2.828$ is the number of standard errors between that estimate and zero. The critical value on two degrees of freedom is about $4.303$, so the same estimate would have been rejected in a larger sample with the same standard error. The interval for $b_1$,

$$
2 \pm t_{2,\,0.975}\sqrt{\dfrac{1}{2}},
$$

contains both negative and positive numbers. It reports that uncertainty. It does not replace the estimate by zero.


In [3]:
# The slope interval contains zero and also contains the estimate 2.
import math
import scipy.stats as stats
half = stats.t.ppf(0.975, 2) * math.sqrt(1 / 2)
lower, upper = 2 - half, 2 + half
print("slope interval", lower, upper)
assert lower < 0 < 2 < upper


slope interval -1.0424349222966542 5.042434922296654


## Summary

- $s^2 = 5$, $\operatorname{se}(b_0) = \sqrt{17/7}$, $\operatorname{se}(b_1) = \sqrt{1/2}$, and $\operatorname{se}(b_2) = \sqrt{5/14}$.
- $t_2 = \sqrt{126/5}$ exceeds the $t_2$ critical value. $t_0$ and $t_1$ do not.
- $t_2^2 = 126/5$ is the $F$ statistic of the extra-sum-of-squares test.
